In [ ]:
!pip install -r requirements.txt

In [21]:
import asyncio
import os
import sys
from datetime import datetime

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.filters import FunctionInvocationContext
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.functions import KernelArguments
from semantic_kernel.kernel import Kernel
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
import pyodbc
import aioodbc
from semantic_kernel.functions import kernel_function
from typing import Annotated, Optional

In [2]:
kernel = Kernel()
load_dotenv()
# Add the AzureChatCompletion AI Service to the Kernel
service_id = "openai"
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)
chat_completion = AzureChatCompletion(ad_token_provider=token_provider, deployment_name=os.environ["AOAI_DEPLOYMENT_NAME"], endpoint=os.environ["AOAI_ENDPOINT"], api_version=os.environ["AOAI_API_VERSION"], service_id=service_id)
kernel.add_service(chat_completion)

In [10]:
driver = os.environ["SQL_DRIVER"]
server = os.environ["SQL_SERVER"]
database = os.environ["SQL_DATABASE"]
user = os.environ["SQL_USERNAME"]
password = os.environ["SQL_PASSWORD"]
conn_string = f"Driver={driver};Server={server},1433;Database={database};Uid={user};Pwd={password};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=60;"
table_name = "product_pricing"
conn = pyodbc.connect(conn_string)

In [12]:
import pandas as pd
import pyodbc

# Load the CSV file into a Pandas DataFrame
csv_file_path = "./data/pricing.csv"  # Replace with the path to your CSV file
df = pd.read_csv(csv_file_path)

# Rename a column to 'timestamp' and convert it to datetime

print(df.head(5))

# Function to map Pandas data types to SQL Server data types
def map_dtype_to_sql(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BIT"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATETIME"
    else:
        return "NVARCHAR(MAX)"  # Default to NVARCHAR for strings and other types

# Connect to the database
try:
    conn = pyodbc.connect(conn_string)
    cursor = conn.cursor()
    print("Connected to the database successfully.")

    # Create a table dynamically based on the DataFrame columns and their data types

    columns = ", ".join([f"[{col.lower()}] {map_dtype_to_sql(dtype)}" for col, dtype in df.dtypes.items()])
    create_table_query = f"CREATE TABLE {table_name} ({columns});"

    # print(create_table_query)
    # Drop the table if it already exists (optional)
    cursor.execute(f"IF OBJECT_ID('{table_name}', 'U') IS NOT NULL DROP TABLE {table_name};")
    cursor.execute(create_table_query)
    print(f"Table '{table_name}' created successfully.")

    batch_size = 10  # Define the batch size
    rows = [tuple(row) for row in df.itertuples(index=False, name=None)]  # Convert DataFrame rows to tuples

    try:
        for i in range(0, len(rows), batch_size):
            batch = rows[i:i + batch_size]  # Get a batch of rows
            placeholders = ", ".join(["?"] * len(df.columns))  # Create placeholders for the query
            insert_query = f"INSERT INTO {table_name} VALUES ({placeholders})"
            cursor.executemany(insert_query, batch)  # Execute the batch insert
            conn.commit()  # Commit the batch
            print(f"Inserted rows {i} to {i + len(batch) - 1} successfully.")

        print("All data inserted successfully.")

    except Exception as e:
        print(f"An error occurred during batch insert: {e}")
    
    conn.commit()
    print("Data inserted successfully.")

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    if 'conn' in locals():
        conn.close()
        print("Database connection closed.")

     id       category                 product_name   variant  \
0  T001  camping_tents             all_weather_tent  2_person   
1  T002  camping_tents             all_weather_tent  4_person   
2  T003  camping_tents             all_weather_tent  6_person   
3  T004  camping_tents  ultralight_backpacking_tent  1_person   
4  T005  camping_tents  ultralight_backpacking_tent  2_person   

                                   features  price_usd  
0             waterproof, wind_uv_resistant        150  
1             waterproof, wind_uv_resistant        220  
2             waterproof, wind_uv_resistant        280  
3  lightweight, quick_setup, ripstop_fabric        180  
4  lightweight, quick_setup, ripstop_fabric        195  
Connected to the database successfully.
Table 'product_pricing' created successfully.
Inserted rows 0 to 9 successfully.
Inserted rows 10 to 19 successfully.
Inserted rows 20 to 29 successfully.
Inserted rows 30 to 35 successfully.
All data inserted successfully.
Dat

In [ ]:
conn = pyodbc.connect(conn_string)
column_info = []
cur = conn.cursor()
cur.execute(f"SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '{table_name}'")
columns = cur.fetchall()
    # col[1] is the column name, col[2] is the column type
column_info = [f"{col[0]}: {col[1]}" for col in columns]

In [24]:
conn = pyodbc.connect(conn_string)
cur = conn.cursor()
cur.execute(f"SELECT DISTINCT category, product_name from {table_name}")
columns = cur.fetchall()
    # col[1] is the column name, col[2] is the column type
unique_product_category = [f"{col[0]}: {col[1]}" for col in columns]

In [27]:

"""Return a string containing the database schema information and common query fields."""
table_dicts = []
table_dicts.append({"table_name": table_name, "column_names": column_info})

database_info = "\n".join(
    [
        f"Table {table['table_name']} Schema: Columns: {', '.join(table['column_names'])}"
        for table in table_dicts
    ]
)
products = unique_product_category

database_info += f"\n Unique category: products combination - {', '.join(unique_product_category)}"
database_info += "\n\n"

In [19]:
# Define the auto function invocation filter that will be used by the kernel
async def function_invocation_filter(context: FunctionInvocationContext, next):
    """A filter that will be called for each function call in the response."""
    if "messages" not in context.arguments:
        await next(context)
        return
    print(f"    Agent [{context.function.name}] called with messages: {context.arguments['messages']}")
    await next(context)
    print(f"    Response from agent [{context.function.name}]: {context.result.value}")

kernel.add_filter("function_invocation", function_invocation_filter)


In [33]:
import json
class PricingPlugin:
   
    """A sample Menu Plugin used for the concept sample."""

    @kernel_function(description="This function is used to answer user questions about product pricing and inventory by executing SQL queries against the database.")
    async def execute_sql(self, sql_query: Annotated[str, "The input should be a well-formed SQL query to extract information based on the user's question. The query result will be returned as a JSON object."]) -> Annotated[str, "Return data in JSON serializable format"]:
        try:
            conn = await aioodbc.connect(dsn=conn_string)
            # Perform the query asynchronously
            cur = await conn.cursor()
            await cur.execute(sql_query)
            rows = await cur.fetchall()
            columns = [description[0] for description in cur.description]

            if not rows:  # No need to create DataFrame if there are no rows
                return json.dumps("The query returned no results. Try a different question.")
            results = [tuple(row) for row in rows]
            await conn.close()
            data = pd.DataFrame(results, columns=columns)
            # return data.to_dict(orient="records")
            return data.to_json(index=False, orient="split")

        except Exception as e:
            return json.dumps({"SQL query failed with error": str(e), "query": sql_query})


In [30]:
settings = kernel.get_prompt_execution_settings_from_service_id(service_id=service_id)
# Configure the function choice behavior to auto invoke kernel functions
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

# Create the agent
agent = ChatCompletionAgent(
    kernel=kernel,
    name="PRICING_AGENT",
    instructions=f"""
        You are a pricing agent. Your job is to provide pricing information for a given product. 
        Use the pricing database as defined by the schema: {database_info}
        All product names are lowercase with spaces replaced by underscores. 
        """,
    arguments=KernelArguments(settings=settings),
    plugins=[PricingPlugin()],
)

In [31]:
print(agent)

arguments={} description=None id='e31e1492-44e0-46b6-8dec-23ccca590a33' instructions='\n        You are a pricing agent. Your job is to provide pricing information for a given product. \n        Use the pricing database as defined by the schema: Table product_pricing Schema: Columns: id: nvarchar, category: nvarchar, product_name: nvarchar, variant: nvarchar, features: nvarchar, price_usd: int\n Unique category: products combination - accessories: camping_stove, accessories: multi_tool_kit, accessories: sleeping_bag, backpacks: daypack, backpacks: expedition_backpack, backpacks: tactical_backpack, camping_tents: all_weather_tent, camping_tents: family_cabin_tent, camping_tents: ultralight_backpacking_tent, hiking_boots: lightweight_trail_shoes, hiking_boots: mountaineering_boots, hiking_boots: waterproof_hiking_boots, outdoor_apparel: insulated_jacket, outdoor_apparel: quick_dry_pants, outdoor_apparel: thermal_base_layers, outdoor_tech: action_camera, outdoor_tech: gps_navigation_devic

In [35]:
thread: ChatHistoryAgentThread = None
response = await agent.get_response(
        messages="What backpacks are available and their price range",
        thread=thread,
    )
print(f"Response: {response}")

C:\Users\divyesheth\AppData\Local\Temp\ipykernel_13148\2083117470.py:19: RuntimeWarning: coroutine 'Connection.close' was never awaited
  conn.close()
Unclosed connection
connection: <aioodbc.connection.Connection object at 0x0000019A81F76AD0>


Response: Here are the available backpacks and their prices:

- **Expedition Backpack**: $160
- **Daypack**: $54
- **Tactical Backpack**: $98

The price range for backpacks is from $54 to $160.
